In [ ]:

from pathlib import Path
import pandas as pd

# Find the raw data folder
data_folder = Path("../data/raw")

# Find all CSV files inside it
files = list(data_folder.glob("*.csv"))

print("CSV files found:", len(files))

# Load each CSV file
datasets = {}

for file in files:
    df = pd.read_csv(file)
    datasets[file.stem] = df

    print("\nFile:", file.name)
    print("Rows and columns:", df.shape)
    print("Column names:")
    print(df.columns.tolist())


In [ ]:

for name, df in datasets.items():
    print("\nDataset:", name)

    display(df.head())
    display(df.info())
    display(df.isnull().sum())


In [ ]:

for name, df in datasets.items():
    print("\n==========", name, "==========")

    # Convert Date from text to actual dates
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

    # Check the date range
    print("\nDate range:")
    print("First date:", df["Date"].min())
    print("Last date:", df["Date"].max())
    print("Invalid dates:", df["Date"].isna().sum())

    # Count unique locations, SKUs and categories
    print("\nUnique fulfilment locations:",
          df["Fulfilment Location"].nunique())

    print("Unique SKUs:", df["SKU Code"].nunique())

    print("Unique categories:", df["Category1"].nunique())

    # See the actual names
    print("\nFulfilment locations:")
    print(df["Fulfilment Location"].value_counts().head(10))

    print("\nTop 10 SKUs by number of rows:")
    print(df["SKUName"].value_counts().head(10))

    print("\nCategories:")
    print(df["Category1"].value_counts())


In [ ]:

for name, df in datasets.items():
    print("\n==============================")
    print("DATASET:", name)
    print("==============================")

    print("Rows:", len(df))
    print("Date range:",
          df["Date"].min(), "to", df["Date"].max())

    print("Unique locations:",
          df["Fulfilment Location"].nunique())

    print("Unique SKUs:",
          df["SKU Code"].nunique())

    print("\nCategory1 values:")
    print(df["Category1"].value_counts().head(10))

    print("\nSub Category values:")
    print(df["Sub Category"].value_counts().head(10))

    print("\nSegment values:")
    print(df["Segment"].value_counts().head(10))


In [ ]:
print("Datasets loaded:", list(datasets.keys()))

In [ ]:

for name, df in datasets.items():
    print("\n==========", name, "==========")

    # 1. Check actual product names
    print("\nTop 15 product names:")
    print(df["SKUName"].value_counts().head(15))

    # 2. Check order classifications
    print("\nOrder classifications:")
    print(df["Order Classification"].value_counts())

    # 3. Check sales quantity values
    print("\nNet Sold Quantity summary:")
    print(df["Net Sold Quantity"].describe())

    # 4. Check for duplicate rows
    print("\nExact duplicate rows:", df.duplicated().sum())

    # 5. Check negative net quantities
    print("Negative net quantities:",
          (df["Net Sold Quantity"] < 0).sum())


In [ ]:

for name, df in datasets.items():
    print("\n==========", name, "==========")

    print("Rows:", len(df))
    print("Order classifications:")
    print(df["Order Classification"].value_counts().to_dict())

    print("\nNet quantity statistics:")
    print(df["Net Sold Quantity"].describe().round(2).to_dict())

    print("\nExact duplicate rows:", df.duplicated().sum())
    print("Negative net quantities:",
          (df["Net Sold Quantity"] < 0).sum())

    print("\nTop 5 products by row count:")
    print(df["SKUName"].value_counts().head(5).to_dict())



In [ ]:

import matplotlib.pyplot as plt

for name, df in datasets.items():

    # Keep customer sales orders only
    sales = df[df["Order Classification"] == "SO"].copy()

    # Add up net quantities for each week
    weekly_sales = (
        sales.groupby(pd.Grouper(key="Date", freq="W"))
        ["Net Sold Quantity"]
        .sum()
    )

    # Plot the weekly sales
    plt.figure(figsize=(12, 4))
    plt.plot(weekly_sales.index, weekly_sales.values)

    plt.title("Weekly Net Sold Quantity - " + name)
    plt.xlabel("Week")
    plt.ylabel("Net Sold Quantity")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:

for name, df in datasets.items():

    print("\n==========", name, "==========")

    sales = df[df["Order Classification"] == "SO"]

    # Show rows with net quantities of 50 or more
    large_orders = sales[
        sales["Net Sold Quantity"] >= 50
    ].sort_values(
        "Net Sold Quantity", ascending=False
    )

    print(
        large_orders[
            ["Date", "Fulfilment Location", "OrderNo",
             "SKUName", "Sales Quantity",
             "Cancel Quantity", "Return Quantity",
             "Net Sold Quantity"]
        ].head(10).to_string(index=False)
    )


In [ ]:

for name, df in datasets.items():
    sales = df[df["Order Classification"] == "SO"].copy()

    print("\n==========", name, "==========")

    # Count orders with large net quantities
    for threshold in [10, 50, 100]:
        count = (sales["Net Sold Quantity"] >= threshold).sum()
        print(f"Rows with net quantity >= {threshold}: {count}")

    # Show the largest orders
    large = sales.nlargest(10, "Net Sold Quantity")

    display(large[
        ["Date", "Fulfilment Location", "OrderNo",
         "SKUName", "Sales Quantity", "Net Sold Quantity"]
    ])


In [ ]:

for name, df in datasets.items():
    print("\n==========", name, "==========")

    # Find rows identical across all columns
    duplicates = df[df.duplicated(keep=False)]

    print("Rows involved in exact duplicates:", len(duplicates))

    # Display a few examples
    display(duplicates.head(10))

    # Check repeated order numbers
    print(
        "Rows with repeated order numbers:",
        df["OrderNo"].duplicated().sum()
    )


In [ ]:

for name, df in datasets.items():
    print("\n==========", name, "==========")

    # Count all rows
    print("Original rows:", len(df))

    # Count rows after removing exact copies temporarily
    no_duplicates = df.drop_duplicates()
    print("Rows after removing exact duplicates:",
          len(no_duplicates))

    # Compare total net quantities
    print("Original total net quantity:",
          df["Net Sold Quantity"].sum())

    print("Total after removing exact duplicates:",
          no_duplicates["Net Sold Quantity"].sum())

    print("Quantity difference:",
          df["Net Sold Quantity"].sum()
          - no_duplicates["Net Sold Quantity"].sum())


In [ ]:

for name, df in datasets.items():
    print("\n==========", name, "==========")

    # Check whether net quantity equals sales minus cancellations
    # plus returns
    calculated_net = (
        df["Sales Quantity"]
        - df["Cancel Quantity"]
        - df["Return Quantity"]
    )

    mismatch = calculated_net != df["Net Sold Quantity"]

    print("Rows where calculated net differs:",
          mismatch.sum())

    # Show examples if there are differences
    if mismatch.any():
        display(df.loc[mismatch, [
            "Order Classification",
            "Sales Quantity",
            "Cancel Quantity",
            "Return Quantity",
            "Net Sold Quantity"
        ]].head(10))


In [ ]:

for name, df in datasets.items():
    print("\n==========", name, "==========")

    print("Missing dates:", df["Date"].isna().sum())

    print("Missing SKU codes:", df["SKU Code"].isna().sum())

    print("Missing fulfilment locations:",
          df["Fulfilment Location"].isna().sum())

    print("Zero net quantities:",
          (df["Net Sold Quantity"] == 0).sum())

    print("Negative sales quantities:",
          (df["Sales Quantity"] < 0).sum())

    print("Negative prices:", (df["Price"] < 0).sum())

    print("Negative discounts:", (df["Discount"] < 0).sum())


In [ ]:

for name, df in datasets.items():
    print("\n==========", name, "==========")

    # Inspect rows with zero net quantity
    zero_rows = df[df["Net Sold Quantity"] == 0]

    print("\nZero net quantity rows:", len(zero_rows))
    print("Order classifications:")
    print(zero_rows["Order Classification"].value_counts())

    print("\nExamples of zero net quantity rows:")
    display(zero_rows[
        ["Order Classification", "Sales Quantity",
         "Cancel Quantity", "Return Quantity",
         "Net Sold Quantity"]
    ].head(5))

    # Inspect negative discounts
    negative_discount = df[df["Discount"] < 0]

    print("\nNegative discount rows:", len(negative_discount))

    if not negative_discount.empty:
        display(negative_discount[
            ["Date", "OrderNo", "SKUName", "Price", "Discount"]
        ])


In [ ]:

from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")

for file in processed_folder.glob("*_cleaned.csv"):
    df = pd.read_csv(file, low_memory=False)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

    print("\nFile:", file.name)
    print("Rows and columns:", df.shape)
    print("Missing dates:", df["Date"].isna().sum())
    print("Missing SKU codes:", df["SKU Code"].isna().sum())
    print("Missing fulfilment locations:", df["Fulfilment Location"].isna().sum())
    print("Missing net quantities:", df["Net Sold Quantity"].isna().sum())
    print("Duplicate rows remaining:", df.duplicated().sum())
    print("Order classifications:", df["Order Classification"].unique())


In [ ]:

from pathlib import Path
import pandas as pd

raw_folder = Path("../data/raw")

for file in raw_folder.glob("*.csv"):
    df = pd.read_csv(file, low_memory=False)

    print("\nFile:", file.name)
    print("Original date data type:", df["Date"].dtype)
    print("Sample original dates:")
    print(df["Date"].dropna().astype(str).head(10).to_list())

    parsed_dates = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

    print("Dates that failed parsing:", parsed_dates.isna().sum())
    print("Examples of failed dates:")
    print(df.loc[parsed_dates.isna(), "Date"].head(10).to_list())


In [ ]:

from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")

for file in processed_folder.glob("*_cleaned.csv"):
    df = pd.read_csv(file, low_memory=False)

   
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


    print("\nFile:", file.name)
    print("Missing dates:", df["Date"].isna().sum())
    print("Date range:", df["Date"].min(), "to", df["Date"].max())


In [ ]:

from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")

for file in processed_folder.glob("*_cleaned.csv"):
    df = pd.read_csv(file, low_memory=False)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    df["Week"] = df["Date"].dt.to_period("W").dt.start_time

    weekly_sales = (
        df.groupby(["Week", "Fulfilment Location", "SKU Code"], as_index=False)
        ["Net Sold Quantity"]
        .sum()
    )

    print("\nFile:", file.name)
    print("Weekly rows:", len(weekly_sales))
    print(weekly_sales.head())

    output_file = processed_folder / f"{file.stem}_weekly.csv"
    weekly_sales.to_csv(output_file, index=False)
    print("Saved to:", output_file)


In [ ]:

from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")

for file in processed_folder.glob("*_weekly.csv"):
    df = pd.read_csv(file)
    df["Week"] = pd.to_datetime(df["Week"])

    print("\nFile:", file.name)
    print("First week:", df["Week"].min())
    print("Last week:", df["Week"].max())
    print("Unique weeks:", df["Week"].nunique())
    print("Unique fulfilment locations:", df["Fulfilment Location"].nunique())
    print("Unique SKUs:", df["SKU Code"].nunique())
    print("Location-SKU combinations:",
          df[["Fulfilment Location", "SKU Code"]].drop_duplicates().shape[0])


In [ ]:

from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")

for file in processed_folder.glob("*_weekly.csv"):
    df = pd.read_csv(file)
    df["Week"] = pd.to_datetime(df["Week"])

    total_missing_weeks = 0
    total_expected_rows = 0

    for _, group in df.groupby(["Fulfilment Location", "SKU Code"]):
        first_week = group["Week"].min()
        last_week = group["Week"].max()

        expected_weeks = len(
            pd.date_range(first_week, last_week, freq="W-MON")
        )

        total_expected_rows += expected_weeks
        total_missing_weeks += expected_weeks - len(group)

    print("\nFile:", file.name)
    print("Expected rows within each combination's active period:",
          total_expected_rows)
    print("Missing weeks within active periods:", total_missing_weeks)


In [ ]:

from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")

for file in processed_folder.glob("*_weekly.csv"):
    df = pd.read_csv(file)
    df["Week"] = pd.to_datetime(df["Week"])

    all_rows = []

    for (location, sku), group in df.groupby(
        ["Fulfilment Location", "SKU Code"]
    ):
        group = group.set_index("Week").sort_index()

        full_weeks = pd.date_range(
            start=group.index.min(),
            end=group.index.max(),
            freq="W-MON"
        )

        group = group.reindex(full_weeks)
        group.index.name = "Week"

        group["Fulfilment Location"] = location
        group["SKU Code"] = sku
        group["Net Sold Quantity"] = (
            group["Net Sold Quantity"].fillna(0)
        )

        all_rows.append(group.reset_index())

    complete_df = pd.concat(all_rows, ignore_index=True)

    output_file = processed_folder / file.name.replace(
        "_weekly.csv", "_weekly_filled.csv"
    )
    complete_df.to_csv(output_file, index=False)

    print("\nFile:", output_file.name)
    print("Rows before:", len(df))
    print("Rows after:", len(complete_df))
    print("Missing quantities:", complete_df["Net Sold Quantity"].isna().sum())


In [ ]:

from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")

for file in processed_folder.glob("*_weekly_filled.csv"):
    df = pd.read_csv(file)

    print("\nFile:", file.name)
    print("Total net quantity:", df["Net Sold Quantity"].sum())
    print("Weeks with zero net sales:",
          (df["Net Sold Quantity"] == 0).sum())
    print("Weeks with negative net sales:",
          (df["Net Sold Quantity"] < 0).sum())
    print("Average weekly quantity:",
          round(df["Net Sold Quantity"].mean(), 2))
    print("Maximum weekly quantity:",
          df["Net Sold Quantity"].max())


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

processed_folder = Path("../data/processed")

for file in processed_folder.glob("*_weekly_filled.csv"):
    df = pd.read_csv(file)
    df["Week"] = pd.to_datetime(df["Week"])

    weekly_total = (
        df.groupby("Week")["Net Sold Quantity"]
        .sum()
        .sort_index()
    )

    plt.figure(figsize=(12, 5))
    plt.plot(weekly_total.index, weekly_total.values)
    plt.title(f"Weekly Net Sales - {file.stem}")
    plt.xlabel("Week")
    plt.ylabel("Net Sold Quantity")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:

from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")

for file in processed_folder.glob("*_weekly_filled.csv"):
    df = pd.read_csv(file)
    df["Week"] = pd.to_datetime(df["Week"])

    weekly_total = (
        df.groupby("Week")["Net Sold Quantity"]
        .sum()
        .sort_index()
    )
    print("\nFile:", file.name)
    print("Missing weeks:", df["Week"].isna().sum())
    print("Week range:", df["Week"].min(), "to", df["Week"].max())

    print("\nLast 10 weeks:")
    print(weekly_total.tail(10))

    print("\nTop 5 weeks by net quantity:")
    print(weekly_total.nlargest(5))


## Step 2: Validate Weekly Data

In [ ]:
from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")

files = list(processed_folder.glob("*_weekly_filled.csv"))

print("Weekly-filled files found:", len(files))

for file in files:
    df = pd.read_csv(file, low_memory=False)

    print("\nDataset:", file.name)
    print("Total rows:", len(df))

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nUnique weeks:", df["Week"].nunique())
    print("Unique locations:", df["Fulfilment Location"].nunique())
    print("Unique SKUs:", df["SKU Code"].nunique())

    print("\nQuantity summary:")
    print(df["Net Sold Quantity"].describe())

    print("\nNegative quantities:",
          (df["Net Sold Quantity"] < 0).sum())

    print("Zero-quantity weeks:",
          (df["Net Sold Quantity"] == 0).sum())

In [ ]:
print("\nQuantity summary:")
print(df["Net Sold Quantity"].describe())

print("\nNegative quantities:",
      (df["Net Sold Quantity"] < 0).sum())

print("Zero-quantity weeks:",
      (df["Net Sold Quantity"] == 0).sum())

In [ ]:
from pathlib import Path
import pandas as pd

folder = Path("../data/processed")

for file in folder.glob("*_weekly_filled.csv"):
    df = pd.read_csv(file, low_memory=False)
    qty = df["Net Sold Quantity"]

    print("\n", file.name)
    print("Rows:", len(df))
    print("Zero weeks:", (qty == 0).sum())
    print("Zero weeks (%):", round((qty == 0).mean() * 100, 2))
    print("Maximum quantity:", qty.max())
    print("Weeks:", df["Week"].nunique())

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

folder = Path("../data/processed")

for file in folder.glob("*_weekly_filled.csv"):
    df = pd.read_csv(file, low_memory=False)
    df["Week"] = pd.to_datetime(df["Week"])

    # Total demand across all locations and SKUs per week
    weekly_total = df.groupby("Week")["Net Sold Quantity"].sum()

    plt.figure(figsize=(12, 5))
    plt.plot(weekly_total.index, weekly_total.values)
    plt.title(f"Weekly Net Sold Quantity - {file.stem}")
    plt.xlabel("Week")
    plt.ylabel("Net Sold Quantity")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

folder = Path("../data/processed")

for file in folder.glob("*_weekly_filled.csv"):
    df = pd.read_csv(file, low_memory=False)

    print("\nDataset:", file.name)

    # Show the 10 largest weekly quantities
    largest = df.nlargest(10, "Net Sold Quantity")

    print(largest[
        ["Week", "Fulfilment Location", "SKU Code", "Net Sold Quantity"]
    ].to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd

file = Path("../data/processed/levis_Order_data_cleaned.csv")
df = pd.read_csv(file, low_memory=False)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Inspect the SKU with the largest weekly quantity
sku = "004JH-0072L"

check = df[
    (df["SKU Code"].astype(str) == sku) &
    (df["Fulfilment Location"] == "South Warehouse") &
    (df["Date"] >= "2026-02-02") &
    (df["Date"] < "2026-02-09")
]

print("Transaction rows:", len(check))
print("Total net sold quantity:", check["Net Sold Quantity"].sum())
print("Unique orders:", check["OrderNo"].nunique())

print(check[
    ["Date", "OrderNo", "Sales Quantity",
     "Cancel Quantity", "Return Quantity", "Net Sold Quantity"]
].head(20).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd

# Load the cleaned Levi's data
file = Path("../data/processed/levis_Order_data_cleaned.csv")
df = pd.read_csv(file, low_memory=False)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Select the SKU, location, and week we are investigating
check = df[
    (df["SKU Code"].astype(str) == "004JH-0072L") &
    (df["Fulfilment Location"] == "South Warehouse") &
    (df["Date"] >= "2026-02-02") &
    (df["Date"] < "2026-02-09")
].copy()

# Check the largest individual quantities
print("Largest individual row quantities:")
print(
    check.nlargest(10, "Net Sold Quantity")[
        ["Date", "OrderNo", "SKU Code", "Net Sold Quantity"]
    ].to_string(index=False)
)

# Check repeated order numbers
print("\nRepeated order numbers:")
repeated = check[check["OrderNo"].duplicated(keep=False)]

if repeated.empty:
    print("No repeated order numbers.")
else:
    print(
        repeated[["OrderNo", "Date", "Net Sold Quantity"]]
        .sort_values("OrderNo")
        .to_string(index=False)
    )

In [ ]:
print(
    check[check["OrderNo"] == "Z15135086"].to_string(index=False)
)